## **Ensemble Methods**

#### **VOTING CLASSIFIER**
-------------------------

In [1]:
# VOTING CLASSIFER

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

df = pd.read_csv('china_used_cars.csv')

# DROP THE TARGET + COLUMNS THAT LEAK IT DIRECTLY (BATTERY/MOTOR SPECS DEFINE is_electric)

drop_cols = ['is_electric', 'battery_capacity_kwh', 'motor_power_kw',
             'price', 'mileage_km', 'log_mileage', 'mileage_per_year', 'year', 'month']
X = df.drop(columns=drop_cols)
y = df['is_electric']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf1 = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf2 = GradientBoostingClassifier(random_state=42)
clf3 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))  # SCALE FOR LR ONLY

voting_soft = VotingClassifier(
    estimators=[('rf', clf1), ('gb', clf2), ('lr', clf3)],
    voting='soft',
    weights=[2, 1, 1]   # TRUST THE RANDOM FOREST A BIT MORE
)
voting_soft.fit(X_train, y_train)
pred = voting_soft.predict(X_test)
print("Voting accuracy:", accuracy_score(y_test, pred))

# Compare against each individual model
for name, clf in voting_soft.named_estimators_.items():
    p = clf.predict(X_test)
    print(f"{name} accuracy: {accuracy_score(y_test, p):.4f}")

Voting accuracy: 0.992831541218638
rf accuracy: 0.9904
gb accuracy: 0.9916
lr accuracy: 0.9892


#### **VOTING REGRESSOR**
-------------------------

In [ ]:
from sklearn.ensemble import VotingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

drop_cols = ['price', 'mileage_km', 'log_mileage', 'mileage_per_year', 'year', 'month']
X = df.drop(columns=drop_cols)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

r1 = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
r2 = GradientBoostingRegressor(random_state=42)
r3 = make_pipeline(StandardScaler(), LinearRegression())

voting_reg = VotingRegressor(estimators=[('rf', r1), ('gb', r2), ('lr', r3)], weights=[2, 2, 1]) # TRUST MORE ON RANDOM FOREST
voting_reg.fit(X_train, y_train)
pred = voting_reg.predict(X_test)
print("Voting  -> MAE:", mean_absolute_error(y_test, pred), " R2:", r2_score(y_test, pred))

for name, est in voting_reg.named_estimators_.items():
    p = est.predict(X_test)
    print(f"{name:4s} -> MAE: {mean_absolute_error(y_test, p):.2f}  R2: {r2_score(y_test, p):.4f}")

Voting  -> MAE: 17487.119753733  R2: 0.5977990326722968
rf   -> MAE: 12357.65  R2: 0.7044
gb   -> MAE: 19420.69  R2: 0.3796
lr   -> MAE: 34686.28  R2: 0.2235
